#Корпоративный карьерный ассистент

###Бизнес-проблема

В компании карьерная информация о ролях, навыках и компетенциях часто оказывается несистемной и распределена по разным источникам, из-за чего сотрудникам сложно быстро понять, какие требования предъявляются к интересующей их роли, чем отличаются смежные направления и какие навыки необходимо развивать для профессионального роста. Это создает барьеры для осознанного карьерного развития и приводит к тому, что сотрудники дольше ищут ответы на свои вопросы или принимают решения на основе неполной информации.


###Предлагаемое решение
Разработанный бот - это внутренний карьерный ассистент, который отвечает в формате диалога на вопросы сотрудников о ролях, навыках, компетенциях и направлениях профессионального развития на основе внутренней базы знаний. Бот упрощает карьерную навигацию сотрудников и снижает нагрузку на HR и руководителей, которым чаще всего приходится отвечать на типовые вопросы сотрудников. Продукт может быть адаптирован под любые корпоративные модели компетенций.

###Бот
@skills_assistant_bot



In [ ]:
!pip install -q qdrant-client sentence-transformers llama-index openai pyTelegramBotAPI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
# Блок 1. Загружаем секретные ключи и параметры доступа, необходимые для подключения
# к OpenAI, Telegram-боту и Qdrant

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TELEGRAM_BOT_TOKEN"] = userdata.get("TELEGRAM_BOT_TOKEN")
os.environ["QDRANT_URL"] = "https://89879b8d-2f46-427b-9af8-d69677c391b7.us-east4-0.gcp.cloud.qdrant.io:6333"
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")

Секреты загружены


In [ ]:
# Блок 2. Импортируем основные модули, задаем параметры работы системы
# и инициализируем клиенты для подключения к OpenAI и Qdrant

import json
import re
import logging
import threading
from typing import List, Dict, Optional
from collections import defaultdict

import numpy as np
import telebot
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from openai import OpenAI

# Логирование
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("career_bot")

# Параметры
COLLECTION_NAME = "career_assistant_e5"
EMBED_MODEL_NAME = "intfloat/multilingual-e5-base"
VECTOR_SIZE = 768
LLM_MODEL = "gpt-4o-mini"
CHUNK_SIZE = 512
CHUNK_OVERLAP = 80
TOP_K = 5
MAX_AGENT_ITERATIONS = 8
MAX_CLARIFICATIONS = 2
MAX_HISTORY_PAIRS = 8

# Клиенты
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])


Импорты и клиенты готовы


In [ ]:
# Блок 3. Загружаем модель эмбеддингов, которая преобразует текстовые фрагменты и пользовательские запросы
# в векторное представление для последующего семантического поиска

embedder = SentenceTransformer(EMBED_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Эмбеддер загружен


In [ ]:
# Блок 4. Преобразовываем исходный документ с описанием ролей и навыков в базу знаний для RAG-системы:
# текст разбивается на чанки, каждому фрагменту присваивается роль,
# после чего создаются эмбеддинги и данные загружаются в Qdrant

import re
from collections import Counter

# Загрузка документа
documents = SimpleDirectoryReader(input_files=["text2_expanded.txt"]).load_data()
full_text = documents[0].text
print(f"Документов: {len(documents)}")

# Находим роли и их позиции в тексте
roles = []
for match in re.finditer(r"^(Навыки .+)$", full_text, re.MULTILINE):
    roles.append({"name": match.group(1).strip(), "start": match.start()})
print(f"Ролей найдено: {len(roles)}")

# Чанкинг
splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents(documents)
texts = [n.text for n in nodes]
print(f"Чанков: {len(texts)}")

# Определяем роль для каждого чанка
chunk_roles = []
for node in nodes:
    chunk_start = full_text.find(node.text[:50])
    if chunk_start == -1:
        chunk_start = 0
    current_role = "Общее"
    for role in roles:
        if role["start"] <= chunk_start:
            current_role = role["name"]
        else:
            break
    chunk_roles.append(current_role)

print("Чанков по ролям:")
for role, count in Counter(chunk_roles).most_common():
    print(f"  {role}: {count}")

# Создание коллекции
existing = [c.name for c in qdrant.get_collections().collections]
if COLLECTION_NAME in existing:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE),
)
print("Коллекция создана")


# Эмбеддинги с префиксом роли
passages = [f"passage: {role}. {text}" for text, role in zip(texts, chunk_roles)]
vectors = embedder.encode(passages, normalize_embeddings=True, batch_size=64, show_progress_bar=True)
print("Эмбеддинги готовы")

# Загрузка в Qdrant с метаданными роли
points = [
    models.PointStruct(
        id=idx,
        vector=vectors[idx].tolist(),
        payload={"text": texts[idx], "chunk_id": idx, "source": "text2_expanded.txt", "role": chunk_roles[idx]},
    )
    for idx in range(len(texts))
]
for i in range(0, len(points), 100):
    qdrant.upsert(collection_name=COLLECTION_NAME, points=points[i:i+100])

print(f"Загружено в Qdrant: {len(points)} точек")



Документов: 1
Ролей найдено: 15
Чанков: 53
Чанков по ролям:
  Навыки бэкенд-разработчиков: 4
  Навыки фронтенд-разработчиков: 4
  Навыки аналитика-разработчика: 4
  Навыки продуктового менеджера: 4
  Навыки проектного менеджера: 4
  Навыки QA-инженера: 4
  Навыки DevOps-инженера: 4
  Навыки мобильного разработчика: 4
  Навыки ML-разработчика: 3
  Навыки дата-инженера: 3
  Навыки UX/UI-дизайнера: 3
  Навыки бизнес-аналитика: 3
  Навыки системного аналитика: 3
  Навыки тимлида: 3
  Навыки технического писателя: 3
Коллекция создана


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Эмбеддинги готовы
Загружено в Qdrant: 53 точек


In [ ]:
# Блок 5. Реализуем механизм поиска по базе знаний: пользовательский запрос расширяется за счет альтернативных формулировок,
# после чего выполняется семантический поиск по векторной базе и отбираются наиболее релевантные фрагменты

def expand_query(query: str) -> List[str]:
    """LLM генерирует альтернативные формулировки поискового запроса."""
    prompt = f"""Ты помогаешь улучшить поиск по базе знаний о карьерных ролях и навыках.

Пользователь ищет: "{query}"

Сгенерируй 2-3 альтернативные формулировки этого запроса,
которые могут использовать другие названия ролей или синонимы.
Например, "продуктовый дизайнер" → "UX/UI дизайнер", "дизайнер интерфейсов".

Верни JSON-список строк. Только JSON, без пояснений."""

    resp = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r"```(?:json)?\s*", "", raw).rstrip("`").strip()
    try:
        variants = json.loads(raw)
        if isinstance(variants, list):
            return [str(v) for v in variants]
    except json.JSONDecodeError:
        pass
    return []


def search_knowledge_base(query: str, top_k: int = TOP_K) -> List[Dict]:
    """
    Поиск с Query Expansion: ищем по оригинальному запросу
    + по альтернативным формулировкам от LLM.
    Результаты объединяются и дедуплицируются по chunk_id.
    """
    # Все варианты запроса: оригинал + расширения
    queries = [query] + expand_query(query)

    seen_ids = set()
    all_hits = []

    for q in queries:
        qvec = embedder.encode([f"query: {q}"], normalize_embeddings=True)[0].tolist()
        results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=qvec,
            limit=top_k,
            with_payload=True,
        ).points

        for r in results:
            cid = r.payload.get("chunk_id", -1)
            if cid not in seen_ids:
                seen_ids.add(cid)
                all_hits.append({
                    "text": r.payload["text"],
                    "chunk_id": cid,
                    "score": r.score,
                    "role": r.payload.get("role", "unknown"),
                })

    # Сортируем по score и берем top_k
    all_hits.sort(key=lambda x: x["score"], reverse=True)
    return all_hits[:top_k]

In [ ]:
# Блок 6. Реализуем агентскую логику бота: задаются доступные инструменты, формулируются правила поведения модели,
# хранится история диалога и описывается основной цикл, в котором агент принимает решение
# (искать информацию в базе знаний, уточнять запрос или формировать ответ пользователю)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": (
                "Поиск по внутренней базе знаний компании о карьерных ролях и навыках. "
                "Используй для поиска информации о навыках, ролях, обязанностях и компетенциях. "
                "Можешь вызывать несколько раз с разными запросами. "
                "Формулируй запросы конкретно."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Поисковый запрос к базе знаний."}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "ask_user_clarification",
            "description": (
                "Задать уточняющий вопрос, если запрос слишком общий. Максимум 2 раза. "
                "Не задавай, если можешь найти ответ через search_knowledge_base."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "Уточняющий вопрос пользователю."}
                },
                "required": ["question"],
            },
        },
    },
]

SYSTEM_PROMPT = """Ты — карьерный ассистент компании. Твоя задача — помогать сотрудникам \
разобраться в навыках, ролях и компетенциях для профессионального развития.

## Главное правило
Ты отвечаешь ТОЛЬКО на основе данных, полученных через инструмент search_knowledge_base. \
Если инструмент не вернул релевантной информации по вопросу — честно скажи, \
что в базе знаний нет данных по этой теме. НЕ достраивай ответ из общих знаний, \
НЕ придумывай навыки и рекомендации, которых нет в результатах поиска.

## Критически важно
НИКОГДА не говори "у меня нет информации" или "в базе нет данных", \
если ты еще не вызвал search_knowledge_base. Сначала ВСЕГДА ищи, \
потом делай выводы. Единственное исключение — явный офтопик \
(вопросы не про карьеру и навыки).

Если пользователь спрашивает про конкретную роль (например, blockchain-разработчик, \
геймдизайнер), а search_knowledge_base вернул данные по ДРУГИМ ролям — \
это НЕ ответ на вопрос. В таком случае честно скажи, что данных по запрошенной \
роли в базе нет, и предложи посмотреть смежные роли, которые есть. \
НЕ переупаковывай навыки одной роли под названием другой.

## Проверка релевантности результатов
После каждого вызова search_knowledge_base задай себе вопрос: \
содержат ли результаты ПРЯМОЕ упоминание той роли, про которую спросил пользователь? \
Если пользователь спросил про "геймдизайнера", а в результатах фигурируют \
"продуктовый дизайнер", "фронтенд-разработчик" — это НЕ релевантные данные. \
Не пытайся адаптировать, экстраполировать или обобщать данные с других ролей. \
Скажи прямо: "В базе знаний нет данных по роли [название]. \
Могу рассказать про смежные роли: [список найденных]."

## Как работать с инструментами
1. Получив вопрос о роли или навыках — СРАЗУ вызывай search_knowledge_base. \
Не пытайся ответить без поиска.
2. Если в вопросе упоминаются ДВЕ или более ролей (сравнение, общие навыки, \
переход между ролями) — ОБЯЗАТЕЛЬНО сделай отдельный поиск по КАЖДОЙ роли. \
Например, на вопрос "чем отличается PM от PjM" сделай два поиска: \
"навыки продуктового менеджера" и "навыки проектного менеджера".
3. Если вопрос слишком общий (например, "как мне развиваться" без указания роли), \
задай уточняющий вопрос через ask_user_clarification. Предложи конкретные варианты.
4. Для поиска формулируй конкретные запросы. Можешь делать несколько запросов \
по разным аспектам одной роли.
5. Не задавай уточняющие вопросы, если можешь найти ответ через поиск.

## Формат ответа
- Адаптируй формат под вопрос: для вопросов о навыках — структурированный список \
с пояснениями; для сравнения ролей — сравнительный анализ; для общих вопросов — \
связный текст.
- Будь конкретным: вместо «развивайте soft skills» пиши, какие именно и зачем.
- Если информация найдена частично — укажи, что именно удалось найти, \
а по каким аспектам данных в базе нет.

## Тон
- Профессиональный, но дружелюбный.
- Конкретный и полезный.
- При неопределенности — уточняй, не угадывай.

## Ограничения
- Ты НЕ отвечаешь на вопросы, не связанные с карьерным развитием, \
навыками и профессиональными ролями. На такие вопросы вежливо объясни, \
что ты специализируешься на карьерном консультировании, и предложи \
задать вопрос по теме.
- Не давай советов по зарплатам, конкретным вакансиям или персональным оценкам.
"""


sessions: Dict[int, List[Dict]] = defaultdict(list)
clarification_counts: Dict[int, int] = defaultdict(int)

def trim_history(chat_id: int):
    history = sessions[chat_id]
    pairs, cut = 0, 0
    for i in range(len(history) - 1, -1, -1):
        if history[i]["role"] == "user":
            pairs += 1
        if pairs > MAX_HISTORY_PAIRS:
            cut = i + 1
            break
    sessions[chat_id] = history[cut:]

def execute_tool(tool_name: str, arguments: dict):
    if tool_name == "search_knowledge_base":
        hits = search_knowledge_base(arguments.get("query", ""))
        if not hits:
            return "Поиск не дал результатов."
        return "\n\n".join(f"[Результат {i+1}, score={h['score']:.3f}]\n{h['text']}" for i, h in enumerate(hits))
    elif tool_name == "ask_user_clarification":
        return None
    return f"Неизвестный инструмент: {tool_name}"

def run_agent(chat_id: int, user_message: str) -> str:
    sessions[chat_id].append({"role": "user", "content": user_message})
    trim_history(chat_id)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + sessions[chat_id]

    for iteration in range(MAX_AGENT_ITERATIONS):
        logger.info(f"Agent iteration {iteration+1}/{MAX_AGENT_ITERATIONS}")
        response = openai_client.chat.completions.create(
            model=LLM_MODEL, messages=messages, tools=TOOLS, tool_choice="auto",
        )
        choice = response.choices[0]
        msg = choice.message

        if msg.tool_calls:
            messages.append({
                "role": "assistant", "content": msg.content or "",
                "tool_calls": [{"id": tc.id, "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                    for tc in msg.tool_calls],
            })
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                logger.info(f"Tool: {fn_name}({fn_args})")

                if fn_name == "ask_user_clarification":
                    if clarification_counts[chat_id] >= MAX_CLARIFICATIONS:
                        messages.append({"role": "tool", "tool_call_id": tc.id,
                            "content": "Лимит уточнений исчерпан. Ответь на основе имеющейся информации."})
                        continue
                    question = fn_args.get("question", "")
                    clarification_counts[chat_id] += 1
                    sessions[chat_id].append({"role": "assistant", "content": question})
                    return question

                result = execute_tool(fn_name, fn_args)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result or ""})
        else:
            answer = msg.content or ""
            sessions[chat_id].append({"role": "assistant", "content": answer})
            clarification_counts[chat_id] = 0
            return answer

    fallback = "Не смог сформировать ответ. Попробуй переформулировать вопрос."
    sessions[chat_id].append({"role": "assistant", "content": fallback})
    clarification_counts[chat_id] = 0
    return fallback



Агент готов


In [ ]:
# Блок 7. Создаем и настраиваем Telegram-бота

bot = telebot.TeleBot(os.environ["TELEGRAM_BOT_TOKEN"])

WELCOME = """Привет! Я карьерный ассистент 🎯

Я помогу тебе разобраться в навыках и компетенциях для различных IT-ролей. Могу рассказать про конкретную роль, сравнить направления или помочь составить план развития.

Примеры вопросов:
• Какие навыки нужны бэкенд-разработчику?
• Чем отличается системный аналитик от бизнес-аналитика?
• Я фронтенд-разработчик, хочу стать техлидом — что развивать?
• Какие навыки общие у ML-инженера и дата-инженера?

Просто напиши свой вопрос!


"""

def safe_reply(message, text):
    parts = [text[i:i+4000] for i in range(0, len(text), 4000)] if text else [""]
    for part in parts:
        try:
            bot.reply_to(message, part, parse_mode="Markdown")
        except:
            bot.reply_to(message, part)

@bot.message_handler(commands=["start", "help"])
def handle_start(message):
    sessions[message.chat.id] = []
    clarification_counts[message.chat.id] = 0
    safe_reply(message, WELCOME)

@bot.message_handler(commands=["reset"])
def handle_reset(message):
    sessions[message.chat.id] = []
    clarification_counts[message.chat.id] = 0
    safe_reply(message, "История сброшена. Задай новый вопрос!")

@bot.message_handler(func=lambda m: True, content_types=["text"])
def handle_text(message):
    q = (message.text or "").strip()
    if not q:
        return
    try:
        bot.send_chat_action(message.chat.id, "typing")
        response = run_agent(message.chat.id, q)
        safe_reply(message, response)
    except Exception as e:
        logger.exception("Ошибка")
        safe_reply(message, "Произошла ошибка. Попробуй еще раз.")

bot.infinity_polling(skip_pending=True, timeout=60, long_polling_timeout=60)

Запуск бота...


2026-03-22 18:43:16,952 (__init__.py:1121 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
ERROR:TeleBot:Infinity polling: polling exited
2026-03-22 18:43:16,956 (__init__.py:1123 MainThread) ERROR - TeleBot: "Break infinity polling"
ERROR:TeleBot:Break infinity polling


In [ ]:
# БЛОК 8. Проводим аудит агентской логики
# Проверяем ключевое свойство Agentic RAG: способность LLM адаптировать стратегию под тип вопроса.
#
# Что логируем для каждого запроса:
# - список вызванных инструментов и их аргументы
# - количество вызовов search_knowledge_base
# - был ли вызов ask_user_clarification
# - количество итераций агентского цикла
# - финальный ответ
#
# Как оцениваем:
# - по паттерну вызовов инструментов
# - по содержанию ответа (для hallucination probe)

import json
from typing import List, Dict
from collections import defaultdict


def _is_text_clarification(answer: str) -> bool:
    """
    Проверяет, содержит ли текст ответа уточняющий вопрос.
    Агент может уточнять не через инструмент, а прямо в тексте —
    это тоже корректное поведение.
    """
    answer_lower = answer.lower()
    markers = [
        "уточните", "уточни", "какую роль", "какое направление",
        "какую сферу", "какая у вас роль", "какая у тебя роль",
        "в каком направлении", "конкретнее", "подробнее",
        "что именно", "какой области", "какую область",
        "пожалуйста, укажите", "пожалуйста, укажи",
        "например,",  # предлагает варианты
    ]
    # Должен быть маркер уточнения И не должно быть развернутого ответа с навыками
    has_clarification = any(m in answer_lower for m in markers)
    # Если ответ короткий (< 500 символов) — скорее всего это уточнение, а не ответ
    is_short = len(answer) < 500
    return has_clarification and is_short



# 8.1. Тестовые сценарии
AGENT_TEST_CASES = [

    # Категория 1: Конкретный вопрос
    # Ожидание: агент вызывает search 1 раз, отвечает на основе результатов
    {
        "query": "Какие навыки нужны DevOps-инженеру?",
        "category": "single_search",
        "description": "Конкретная роль — ожидаем 1 поиск по базе",
        "check": lambda r: r["num_searches"] >= 1 and r["clarification_asked"] is None,
    },
    {
        "query": "Что QA-инженер должен знать про API-тестирование?",
        "category": "single_search",
        "description": "Конкретная роль + аспект — ожидаем поиск по базе",
        "check": lambda r: r["num_searches"] >= 1 and r["clarification_asked"] is None,
    },

    # Категория 2: Сравнительный вопрос
    # Ожидание: агент вызывает search 2+ раза с разными запросами
    {
        "query": "Чем отличается продуктовый менеджер от проектного менеджера?",
        "category": "multi_search",
        "description": "Сравнение двух ролей — ожидаем 2+ поиска",
        "check": lambda r: r["num_searches"] >= 2,
    },
    {
        "query": "Какие навыки общие у фронтенд-разработчика и мобильного разработчика?",
        "category": "multi_search",
        "description": "Поиск общего — ожидаем 2+ поиска по разным ролям",
        "check": lambda r: r["num_searches"] >= 2,
    },

    # Категория 3: Слишком общий, размытый вопрос
    # Ожидание: агент уточняет — либо через инструмент ask_user_clarification,
    # либо задает уточняющий вопрос прямо в тексте ответа
    {
        "query": "Как мне развиваться?",
        "category": "clarification",
        "description": "Нет роли, нет контекста — ожидаем уточнение (через tool или в тексте)",
        "check": lambda r: (
            r["clarification_asked"] is not None
            or _is_text_clarification(r["final_answer"])
        ),
    },
    {
        "query": "Что мне изучить?",
        "category": "clarification",
        "description": "Общий вопрос без деталей — ожидаем уточнение",
        "check": lambda r: (
            r["clarification_asked"] is not None
            or _is_text_clarification(r["final_answer"])
        ),
    },

    # Категория 4: Офтопик
    # Ожидание: агент не вызывает search, а отвечает отказом
    {
        "query": "Какая сегодня погода в Москве?",
        "category": "offtopic",
        "description": "Офтопик — ожидаем отказ БЕЗ обращения к базе",
        "check": lambda r: r["num_searches"] == 0 and r["clarification_asked"] is None,
    },
    {
        "query": "Напиши мне стихотворение про кота",
        "category": "offtopic",
        "description": "Офтопик — ожидаем отказ БЕЗ обращения к базе",
        "check": lambda r: r["num_searches"] == 0 and r["clarification_asked"] is None,
    },

    # Категория 5: Провокация на галлюцинацию
    # Ожидание: агент ходит в базу, но не выдумывает ответ (если роли нет в базе, то он должен честно сказать)
    {
        "query": "Какие навыки нужны blockchain-разработчику?",
        "category": "hallucination_probe",
        "description": "Роли нет в базе — ожидаем поиск + честный ответ",
        "check": lambda r: r["num_searches"] >= 1 and _has_no_data_signal(r["final_answer"]),
    },
    {
        "query": "Расскажи про навыки геймдизайнера",
        "category": "hallucination_probe",
        "description": "Роли нет в базе — не должен выдумывать навыки",
        "check": lambda r: r["num_searches"] >= 1 and _has_no_data_signal(r["final_answer"]),
    },
]


def _has_no_data_signal(answer: str) -> bool:
    """
    Проверяет, содержит ли ответ сигнал об отсутствии данных.
    Агент честно говорит, что не нашел информации — это НЕ галлюцинация.
    """
    answer_lower = answer.lower()
    signals = [
        "нет данных", "не найд", "нет информации", "отсутствует в базе",
        "не содержит", "нет в базе", "не нашел", "не нашел",
        "недостаточно информации", "нет сведений", "не располагаю",
        "база знаний не содержит", "к сожалению",
        "не имею информации", "данных по", "не удалось найти",
        "нет конкретной информации",
        "нет специфической информации",
        "не нашлось",
        "информации о навыках",
        "отсутствуют в базе",
        "не представлен",
    ]
    return any(s in answer_lower for s in signals)



def run_agent_with_audit(query: str, chat_id: int = 99999) -> Dict:
    """
    Запускает агента и собирает полный лог поведения.
    Возвращает словарь с:
      - query, final_answer
      - tool_calls: список всех вызовов [{tool, args}, ...]
      - search_queries: список запросов к search_knowledge_base
      - num_searches: количество вызовов search
      - clarification_asked: текст уточняющего вопроса или None
      - iterations: сколько итераций агентского цикла
    """

    sessions[chat_id] = []
    clarification_counts[chat_id] = 0

    tool_log = []
    search_queries = []
    clarification_asked = None
    iterations_used = 0

    sessions[chat_id].append({"role": "user", "content": query})
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + sessions[chat_id]

    for iteration in range(MAX_AGENT_ITERATIONS):
        iterations_used = iteration + 1

        response = openai_client.chat.completions.create(
            model=LLM_MODEL, messages=messages, tools=TOOLS, tool_choice="auto",
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append({
                "role": "assistant", "content": msg.content or "",
                "tool_calls": [
                    {"id": tc.id, "type": "function",
                     "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                    for tc in msg.tool_calls
                ],
            })

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                tool_log.append({"tool": fn_name, "args": fn_args})

                if fn_name == "search_knowledge_base":
                    search_queries.append(fn_args.get("query", ""))
                    result = execute_tool(fn_name, fn_args)
                    messages.append({
                        "role": "tool", "tool_call_id": tc.id,
                        "content": result or "",
                    })

                elif fn_name == "ask_user_clarification":
                    clarification_asked = fn_args.get("question", "")

                    messages.append({
                        "role": "tool", "tool_call_id": tc.id,
                        "content": "Пользователь не ответил. Ответь на основе имеющейся информации.",
                    })
        else:
            final_answer = msg.content or ""
            sessions[chat_id] = []
            return {
                "query": query,
                "final_answer": final_answer,
                "tool_calls": tool_log,
                "search_queries": search_queries,
                "num_searches": len(search_queries),
                "clarification_asked": clarification_asked,
                "iterations": iterations_used,
            }

    sessions[chat_id] = []
    return {
        "query": query,
        "final_answer": "[цикл исчерпан]",
        "tool_calls": tool_log,
        "search_queries": search_queries,
        "num_searches": len(search_queries),
        "clarification_asked": clarification_asked,
        "iterations": iterations_used,
    }


def run_agent_audit():
    """Запуск аудита агентской логики."""

    print("Аудит агентской логики")
    print()
    print("Проверяемые типы поведения:")
    print("  single_search      конкретный вопрос, 1 поиск")
    print("  multi_search       сравнение/составной, 2+ поиска")
    print("  clarification      общий вопрос, уточнение у пользователя")
    print("  offtopic           не по теме, отказ без поиска")
    print("  hallucination      роли нет в базе, поиск + честный ответ")
    print()

    results = []
    passed = 0
    total = len(AGENT_TEST_CASES)

    for i, case in enumerate(AGENT_TEST_CASES):
        print(f"[{i+1}/{total}] {case['category']}")
        print(f"  Запрос:    {case['query']}")
        print(f"  Ожидание:  {case['description']}")

        try:
            result = run_agent_with_audit(case["query"])
            check_passed = case["check"](result)

            if check_passed:
                passed += 1
                status = "PASS"
            else:
                status = "FAIL"

            behavior_parts = []
            if result["num_searches"] > 0:
                behavior_parts.append(f"{result['num_searches']} поиск(ов)")
            if result["clarification_asked"]:
                behavior_parts.append("уточнение")
            if not behavior_parts:
                behavior_parts.append("прямой ответ без инструментов")
            behavior_str = ", ".join(behavior_parts)

            print(f"  Результат: {status}")
            print(f"  Поведение: {behavior_str}, {result['iterations']} итерац.")

            if result["search_queries"]:
                for j, sq in enumerate(result["search_queries"], 1):
                    print(f"    search #{j}: \"{sq}\"")
            if result["clarification_asked"]:
                print(f"    clarification: \"{result['clarification_asked']}\"")

            print(f"  Ответ:     {result['final_answer'][:150]}")
            print()

            result["test_category"] = case["category"]
            result["test_passed"] = check_passed
            results.append(result)

        except Exception as e:
            print(f"  ОШИБКА: {e}")
            print()
            results.append({
                "query": case["query"],
                "test_category": case["category"],
                "test_passed": False,
                "error": str(e),
            })

    # Сводка
    print(f"Результат: {passed}/{total} тестов пройдено ({100*passed/total:.0f}%)")
    print()

    categories = defaultdict(lambda: {"passed": 0, "total": 0})
    for r in results:
        cat = r["test_category"]
        categories[cat]["total"] += 1
        if r.get("test_passed", False):
            categories[cat]["passed"] += 1

    print("По категориям:")
    for cat in ["single_search", "multi_search", "clarification", "offtopic", "hallucination_probe"]:
        if cat in categories:
            s = categories[cat]
            status = "ok" if s["passed"] == s["total"] else "partial"
            print(f"  {cat:<22} {s['passed']}/{s['total']}  ({status})")

    halluc = [r for r in results if r["test_category"] == "hallucination_probe"]
    if halluc:
        halluc_ok = sum(1 for r in halluc if r.get("test_passed", False))
        print()
        if halluc_ok == len(halluc):
            print(f"Контроль галлюцинаций: {halluc_ok}/{len(halluc)}, агент не галлюцинирует")
        else:
            print(f"Контроль галлюцинаций: {halluc_ok}/{len(halluc)}, есть риск галлюцинаций")

    # Таблица
    print()
    print("Таблица вызовов:")
    print(f"  {'':>6} {'Категория':<22} {'Запрос':<43} {'Поиск':>6} {'Уточн':>6} {'Итер':>5}")
    for r in results:
        q = r.get("query", "?")[:41]
        cat = r.get("test_category", "?")
        ns = r.get("num_searches", "?")
        cl = "да" if r.get("clarification_asked") else "-"
        it = r.get("iterations", "?")
        mark = "pass" if r.get("test_passed") else "FAIL"
        print(f"  [{mark:>4}] {cat:<22} {q:<43} {ns:>6} {cl:>6} {it:>5}")

    return results


# Запуск
audit_results = run_agent_audit()

АУДИТ АГЕНТСКОЙ ЛОГИКИ

Проверяемые типы поведения:
  single_search      конкретный вопрос, 1 поиск
  multi_search       сравнение/составной, 2+ поиска
  clarification      общий вопрос, уточнение у пользователя
  offtopic           не по теме, отказ без поиска
  hallucination      роли нет в базе, поиск + честный ответ

[1/10] single_search
  Запрос:    Какие навыки нужны DevOps-инженеру?
  Ожидание:  Конкретная роль — ожидаем 1 поиск по базе
  Результат: PASS
  Поведение: 1 поиск(ов), 2 итерац.
    search #1: "навыки DevOps-инженера"
  Ответ:     Навыки, необходимые DevOps-инженеру, включают:

1. **Автоматизация и скриптинг**:
   - Автоматизация рутинных операций с помощью скриптов и инструмент

[2/10] single_search
  Запрос:    Что QA-инженер должен знать про API-тестирование?
  Ожидание:  Конкретная роль + аспект — ожидаем поиск по базе
  Результат: PASS
  Поведение: 1 поиск(ов), 2 итерац.
    search #1: "навыки QA-инженера API-тестирование"
  Ответ:     QA-инженеру, занимающемуся 

In [15]:
# БЛОК 9. Оцениваем качество RAG
# 1. Context Precision — какая доля найденных чанков релевантна вопросу
# 2. Context Recall — покрывают ли найденные чанки эталонный ответ
# 3. Faithfulness — все ли утверждения в ответе LLM подтверждаются контекстом

import json
import re
import numpy as np
from typing import List, Dict, Tuple


def _llm_judge(prompt: str) -> str:
    """Вызов LLM-судьи. Возвращает текст ответа."""
    resp = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0,
    )
    return resp.choices[0].message.content.strip()


def _llm_yes_no(prompt: str) -> bool:
    """Вызов LLM-судьи с ответом yes/no."""
    resp = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=5,
        temperature=0,
    )
    return resp.choices[0].message.content.strip().lower().startswith("yes")



# 9.1. Context Precision

def judge_chunk_relevance(question: str, chunk: str) -> bool:
    """LLM оценивает: релевантен ли чанк вопросу?"""
    prompt = f"""Ты оцениваешь качество поиска в RAG-системе.

Вопрос пользователя: {question}

Найденный фрагмент:
{chunk}

Правила оценки:
- Если вопрос про ОДНУ конкретную роль, фрагмент релевантен только если он содержит информацию именно об этой роли
- Если вопрос СРАВНИВАЕТ две роли или спрашивает об ОБЩИХ навыках, фрагмент релевантен если он содержит информацию о ЛЮБОЙ из упомянутых ролей
- Если фрагмент описывает навык, который прямо относится к сути вопроса (даже если название роли не совпадает дословно), считай его релевантным

Содержит ли этот фрагмент информацию, полезную для ответа на вопрос?
Ответь одним словом: yes или no"""
    return _llm_yes_no(prompt)


def context_precision(question: str, chunks: List[str], top_k: int = 5) -> Tuple[float, List[bool]]:
    """
    Context Precision: доля релевантных чанков в top-k, взвешенная по позиции.
    Чанки выше в выдаче важнее (Average Precision).
    """
    if not chunks:
        return 0.0, []

    k = min(top_k, len(chunks))
    flags = []
    relevant_count = 0
    weighted_sum = 0.0

    for i in range(k):
        is_rel = judge_chunk_relevance(question, chunks[i])
        flags.append(is_rel)
        if is_rel:
            relevant_count += 1
            weighted_sum += relevant_count / (i + 1)

    if relevant_count == 0:
        return 0.0, flags
    return min(1.0, weighted_sum / relevant_count), flags



# Context Recall

def judge_statement_supported(question: str, statement: str, chunks: List[str]) -> bool:
    """LLM оценивает: подтверждается ли утверждение найденными чанками?"""
    chunks_text = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))
    prompt = f"""Ты оцениваешь полноту поиска в RAG-системе.

Вопрос: {question}

Утверждение из эталонного ответа:
"{statement}"

Найденные фрагменты:
{chunks_text}

Подтверждается ли это утверждение информацией хотя бы из одного фрагмента?
Ответь одним словом: yes или no"""
    return _llm_yes_no(prompt)


def context_recall(question: str, chunks: List[str], reference_answer: str) -> Tuple[float, int, int]:
    """
    Context Recall: доля утверждений из эталона, подтвержденных контекстом.
    Возвращает (score, supported, total).
    """
    statements = [s.strip() for s in re.split(r"[.;]", reference_answer) if len(s.strip()) > 20]
    if not statements:
        return 0.0, 0, 0

    supported = 0
    for stmt in statements:
        if judge_statement_supported(question, stmt, chunks):
            supported += 1

    return min(1.0, supported / len(statements)), supported, len(statements)



# Faithfulness

def extract_claims(answer: str) -> List[str]:
    """Извлекает фактические утверждения из ответа LLM."""
    prompt = f"""Извлеки все фактические утверждения из следующего текста.
Каждое утверждение должно быть отдельным, проверяемым фактом.
Верни их в формате JSON-списка строк. Только JSON, без пояснений.

Текст:
{answer}"""

    raw = _llm_judge(prompt)

    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"```(?:json)?\s*", "", raw)
        raw = raw.rstrip("`").strip()
    try:
        claims = json.loads(raw)
        if isinstance(claims, list):
            return [str(c) for c in claims if len(str(c)) > 10]
    except json.JSONDecodeError:
        pass

    return [line.strip("- ").strip() for line in raw.split("\n") if len(line.strip()) > 10]


def judge_claim_supported(claim: str, chunks: List[str]) -> bool:
    """LLM оценивает: подтверждается ли утверждение контекстом?"""
    chunks_text = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))
    prompt = f"""Ты проверяешь фактическую точность ответа RAG-системы.

Утверждение из ответа:
"{claim}"

Контекст из базы знаний:
{chunks_text}

Можно ли подтвердить это утверждение информацией из контекста?
Если утверждение содержит информацию, которой НЕТ в контексте — ответь no.
Ответь одним словом: yes или no"""
    return _llm_yes_no(prompt)


def faithfulness(answer: str, chunks: List[str]) -> Tuple[float, int, int, List[str]]:
    """
    Faithfulness: доля утверждений в ответе, подтвержденных контекстом.
    Формула: supported_claims / total_claims

    Возвращает (score, supported, total, unsupported_claims).
    """
    claims = extract_claims(answer)
    if not claims:
        return 1.0, 0, 0, []

    supported = 0
    unsupported = []
    for claim in claims:
        if judge_claim_supported(claim, chunks):
            supported += 1
        else:
            unsupported.append(claim)

    score = supported / len(claims) if claims else 1.0
    return score, supported, len(claims), unsupported



def evaluate_rag(test_file: str = "test_dataset.json", top_k: int = 5):
    """
    Полная оценка RAG: retrieval (precision, recall) + generation (faithfulness).
    Для faithfulness прогоняем агента и получаем реальный ответ.
    """
    with open(test_file, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    all_precision, all_recall, all_faithfulness = [], [], []
    total_llm_calls = 0

    print("ОЦЕНКА КАЧЕСТВА RAG")
    print(f"Метод: LLM-as-a-Judge (без порогов similarity)")
    print(f"Метрики: Context Precision, Context Recall, Faithfulness")
    print(f"Судья: {LLM_MODEL}")
    print(f"Тестовых вопросов: {len(test_data)}, top_k: {top_k}")
    print()

    for i, item in enumerate(test_data):
        question = item["question"]
        ref_answer = item["reference_answer"]

        # Retrieval
        hits = search_knowledge_base(question, top_k=top_k)
        chunks = [h["text"] for h in hits]
        scores = [h["score"] for h in hits]

        # Generation (прогоняем агента для faithfulness)
        sessions[99998] = []
        clarification_counts[99998] = 0
        agent_answer = run_agent(99998, question)
        sessions[99998] = []

        # Метрики
        p, p_flags = context_precision(question, chunks, top_k)
        r, r_sup, r_total = context_recall(question, chunks, ref_answer)
        f, f_sup, f_total, f_unsupported = faithfulness(agent_answer, chunks)

        all_precision.append(p)
        all_recall.append(r)
        all_faithfulness.append(f)

        n_calls = len(p_flags) + r_total + f_total + 1  # +1 за extract_claims
        total_llm_calls += n_calls

        print(f"[{i+1}/{len(test_data)}] {question}")
        print(f"  Precision:   {p:.3f}  ({sum(p_flags)}/{len(p_flags)} чанков релевантны)")
        print(f"  Recall:      {r:.3f}  ({r_sup}/{r_total} утверждений покрыты)")
        print(f"  Faithfulness: {f:.3f}  ({f_sup}/{f_total} утверждений подтверждены)")
        print(f"  Qdrant:      {[f'{s:.3f}' for s in scores[:3]]}")

        # Чанки с метками
        for j, text in enumerate(chunks[:top_k]):
            if j < len(p_flags):
                mark = "rel" if p_flags[j] else "---"
            else:
                mark = "   "
            print(f"    [{mark}] {text[:75]}")

        # Неподтвержденные утверждения
        if f_unsupported:
            print(f"  Неподтвержденные утверждения:")
            for claim in f_unsupported[:3]:
                print(f"    ! {claim[:100]}")
        print()

    # Средние
    avg_p = float(np.mean(all_precision))
    avg_r = float(np.mean(all_recall))
    avg_f = float(np.mean(all_faithfulness))

    print(f"Результат ({len(test_data)} вопросов, ~{total_llm_calls} LLM-вызовов):")
    print(f"  Context Precision:  {avg_p:.4f}  {'ok' if avg_p >= 0.8 else 'ниже цели'} (цель >= 0.8)")
    print(f"  Context Recall:     {avg_r:.4f}  {'ok' if avg_r >= 0.75 else 'ниже цели'} (цель >= 0.75)")
    print(f"  Faithfulness:       {avg_f:.4f}  {'ok' if avg_f >= 0.8 else 'ниже цели'} (цель >= 0.8)")
    print()

    # Таблица
    print("Таблица:")
    print(f"  {'N':>3} {'Prec':>6} {'Rec':>6} {'Faith':>6}  Вопрос")
    for i, (p, r, f, item) in enumerate(zip(all_precision, all_recall, all_faithfulness, test_data)):
        print(f"  {i+1:>3} {p:>6.3f} {r:>6.3f} {f:>6.3f}  {item['question'][:50]}")

    return {
        "context_precision": avg_p,
        "context_recall": avg_r,
        "faithfulness": avg_f,
    }


# Запуск
rag_results = evaluate_rag("test_dataset.json", top_k=5)

ОЦЕНКА КАЧЕСТВА RAG
Метод: LLM-as-a-Judge (без порогов similarity)
Метрики: Context Precision, Context Recall, Faithfulness
Судья: gpt-4o-mini
Тестовых вопросов: 12, top_k: 5

[1/12] Какие навыки нужны бэкенд-разработчику?
  Precision:   0.917  (3/5 чанков релевантны)
  Recall:      1.000  (1/1 утверждений покрыты)
  Faithfulness: 1.000  (26/26 утверждений подтверждены)
  Qdrant:      ['0.897', '0.890', '0.889']
    [rel] Навыки бэкенд-разработчиков
Бэкенд-разработчик отвечает за серверную часть 
    [rel] Он реализует бизнес-логику, обрабатывает ошибки, пишет поддерживаемый и чит
    [---] Умеет читать метрики и логи, участвует в разборе инцидентов и повышении ста
    [rel] Он применяет кэширование, очереди, балансировку нагрузки и асинхронную обра
    [---] Учитывает рост продукта, переиспользование компонентов и согласованность ар

[2/12] Что должен уметь QA-инженер в области автоматизации тестирования?
  Precision:   1.000  (3/5 чанков релевантны)
  Recall:      1.000  (3/3 утвержд

#Направления развития продукта


**MVP (текущий этап)**

Что сделано:
- Agentic RAG бот в Telegram, 15 ролей в базе знаний
- Query Expansion, контроль галлюцинаций
- Eval: Precision 0.67, Recall 0.85, Faithfulness 0.81, аудит агента 10/10

- Деплой бота на Railway

Ограничения MVP:
- Нет грейдов — отвечает одинаково для джуна и сеньора
- Нет обратной связи от пользователей


**MLP (Minimum Lovable Product)**

Цель: разработать продукт, которым сотрудники хотят пользоваться регулярно

Retrieval:
- Cross-encoder reranker для повышения precision
- Грейды (Junior/Middle/Senior/Lead) как метаданные чанков
  с фильтрацией — бот спрашивает уровень и отвечает целевой

Генерация:
- Стриминг ответа в Telegram (не ждать 5-8 секунд)
- Ссылки на источники ("из раздела Навыки DevOps, блок Мониторинг") для повышения доверия пользователей

Продукт:
- Кнопки "полезно / не полезно" после каждого ответа (или CSI/NPS опросы)

Данные:
- Расширение базы: карьерные треки, описание переходов между ролями
- 30-40 ролей вместо 15


**Growth (масштабирование)**

Цель: разработать инструмент, встроенный в HR-процессы компании

Персонализация:
- Профиль сотрудника (роль, грейд, навыки, цели)
- Третий инструмент агента: get_user_profile()
- Рекомендации с учетом текущего уровня и целевой роли
  ("Ты бэкенд Middle, для Senior тебе не хватает: ...")

Аналитика для HR:
- Дашборд с топ запросами: какие роли и навыки ищут чаще
- Выявление трендов: куда хотят расти сотрудники
- Сигнал для L&D — какие обучения запускать


Интеграция:
- API для встраивания в карьерный портал и HR-систему
- Альтернативные каналы интеграции (например, внутренний мессенджер компании)


**Scale Product**

Цель: разработать платформу для карьерного развития

- Fine-tuning эмбеддера на внутренних данных компании
- Автоматическая переиндексация при обновлении документов
- Multi-tenant: адаптация под разные компании и домены
- SLA: latency p95 < 3 сек, uptime 99.9%